In [15]:
import numpy as np
import pandas as pd
import json
import time
from pathlib import Path
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

from sklearn.metrics.pairwise import cosine_similarity

import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

In [9]:
ARTIFACTS_DIR = Path.cwd().parent / "artifacts"
QDRANT_DIR = Path.cwd().parent / "artifacts" / "qdrant"

In [4]:
embeddings = np.load(ARTIFACTS_DIR / "embeddings.npy")
metadata = pd.read_parquet(ARTIFACTS_DIR / "metadata.parquet")
config = json.load(open(ARTIFACTS_DIR / "embedding_config.json", "r"))

print(f"Embeddings shape: {embeddings.shape}")
print(f"Metadata shape: {metadata.shape}")
print(f"Config: {config}")

Embeddings shape: (248218, 384)
Metadata shape: (248218, 9)
Config: {'embedding_model': 'BAAI/bge-small-en-v1.5', 'embedding_dimension': 384, 'normalized': True, 'batch_size': 128, 'num_documents': 248218, 'text_columns': ['name', 'Chemical Class', 'Habit Forming', 'Therapeutic Class', 'Action Class', 'Substitutes', 'Side Effects', 'Uses']}


In [11]:
COLLECTION_NAME = "mirx_qdrant_integration"
TOP_K = 5
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
EMBEDDING_MODEL = config["embedding_model"]
BATCH_SIZE = 1000

In [12]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Embedding dimension: {embedding_model.get_embedding_dimension()}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2942.31it/s]


Embedding dimension: 384


In [13]:
client = QdrantClient(path = QDRANT_DIR)

existing_collections = [collection.name for collection in client.get_collections().collections]

if COLLECTION_NAME not in existing_collections:
    client.create_collection(
        collection_name = COLLECTION_NAME,
        vectors_config = VectorParams(
            size = embedding_model.get_embedding_dimension(),
            distance = Distance.COSINE
        )
    )

    print(
        f"Created collection: {COLLECTION_NAME}"
    )

else:

    print(
        f"Collection already exists: {COLLECTION_NAME}"
    )

Created collection: mirx_qdrant_integration


In [14]:
collection_info = client.get_collection(collection_name = COLLECTION_NAME)

existing_points = collection_info.points_count

print(f"Existing points in collection: {existing_points}")
print(f"Expected points to add: {len(embeddings)}")

Existing points in collection: 0
Expected points to add: 248218


In [16]:
if existing_points < len(embeddings):

    ingestion_start = time.perf_counter()

    for start in tqdm(range(0, len(embeddings), BATCH_SIZE), desc="Uploading vectors"):

        end = min(start + BATCH_SIZE, len(embeddings))

        points = []

        for idx in range(start, end):

            payload = metadata.iloc[idx].to_dict()

            # Convert NumPy types to Python-compatible values
            payload = {
                key: (
                    value.item()
                    if isinstance(value, np.generic)
                    else value
                )
                for key, value in payload.items()
            }

            points.append(
                PointStruct(
                    id=idx,
                    vector=embeddings[idx].tolist(),
                    payload=payload
                )
            )

        client.upsert(collection_name=COLLECTION_NAME, points=points)

    ingestion_time = (time.perf_counter() - ingestion_start)

    print(f"\nIngestion completed in "f"{ingestion_time:.2f} seconds.")

else:

    print(
        "Qdrant already contains the expected "
        "number of vectors."
    )

Uploading vectors:   8%|▊         | 20/249 [01:19<15:47,  4.14s/it]C:\Users\AbhiramJ\AppData\Local\Temp\ipykernel_10380\938611452.py:33: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 21000 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(collection_name=COLLECTION_NAME, points=points)
Uploading vectors: 100%|██████████| 249/249 [15:32<00:00,  3.75s/it]


Ingestion completed in 932.53 seconds.


In [17]:
collection_info = client.get_collection(collection_name=COLLECTION_NAME)

print("=" * 60)
print("QDRANT COLLECTION")
print("=" * 60)

print("Collection:", COLLECTION_NAME)
print("Points:", collection_info.points_count)
print("Vectors:", len(embeddings))
print("Dimension:", config["embedding_dimension"])

assert (
    collection_info.points_count
    == len(embeddings)
), "Qdrant point count mismatch!"

QDRANT COLLECTION
Collection: mirx_qdrant_integration
Points: 248218
Vectors: 248218
Dimension: 384


In [24]:
def retrieve_v2(query, top_k = TOP_K):
    query_embedding = embedding_model.encode(query, convert_to_numpy = True, normalize_embeddings = True)

    search_result = client.query_points(COLLECTION_NAME, query = query_embedding.tolist(), limit = top_k)

    return search_result.points

In [25]:
query = "What are the alternatives to paracetamol?"

results = retrieve_v2(query, top_k = 5)

for i, result in enumerate(results):

    print("=" * 80)
    print(f"RESULT #{i + 1}")
    print("Score:", result.score)
    print()

    print(result.payload["document"])

RESULT #1
Score: 0.6816654118603798

name: dr best paracetamol 250 oral suspension Chemical Class: P-Aminophenol Derivative Habit Forming: No Therapeutic Class: PAIN ANALGESICS Action Class: Analgesic & Antipyretic-PCM Substitutes: Tifmol Oral Suspension, Moltrex Oral Suspension, Kyomol Suspension, Nettmol 250mg Oral Suspension, Padicaf 250mg Oral Suspension Side Effects: Indigestion, Stomach pain, Nausea, Vomiting Uses: Pain relief, Treatment of Fever
RESULT #2
Score: 0.6807458489554865

name: parafen forte tablet Habit Forming: No Therapeutic Class: PAIN ANALGESICS Substitutes: Answell 400 mg/325 mg Tablet, Bruace 400 mg/325 mg Tablet, Rupar 400 mg/325 mg Tablet, Brufamol Tablet, Zupar 400mg/325mg Tablet Side Effects: Heartburn, Indigestion, Nausea, Stomach pain Uses: Pain relief, Treatment of Fever
RESULT #3
Score: 0.6806400535185027

name: brumol m syrup Habit Forming: No Therapeutic Class: PAIN ANALGESICS Substitutes: Ladex-P Syrup, Emfort P Syrup, Mefnoc P Syrup, Mefnix P Syrup, 

In [26]:
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

llm_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype = "auto", device_map = "auto")

llm_model.eval()

Loading weights: 100%|██████████| 434/434 [00:07<00:00, 59.49it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

In [27]:
SYSTEM_PROMPT = """
You are MIRx, a medical information retrieval assistant.

Your task is to answer questions using ONLY the information
provided in the retrieved context.

Rules:

1. Do not invent information.
2. Do not introduce medical facts that are absent from the context.
3. If the retrieved context is insufficient, say so clearly.
4. Keep the answer concise and well structured.
5. When possible, mention the relevant drug or medical entity.
6. Do not make diagnoses or provide personalized medical treatment.
7. Clearly distinguish information found in the retrieved context
   from uncertainty.
"""

In [29]:
def build_context(results):

    context_parts = []

    for i, result in enumerate(results):

        document = result.payload.get(
            "document",
            ""
        )

        context_parts.append(
            f"""
SOURCE {i + 1}
Similarity: {result.score:.4f}

{document}
"""
        )

    return "\n".join(context_parts)

In [30]:
def generate_with_qwen(query, context):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""
Retrieved medical information:

{context}

User question:

{query}

Using only the retrieved medical information above,
answer the user's question.
"""
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt"
    )

    inputs = inputs.to(next(llm_model.parameters()).device)

    input_length = inputs["input_ids"].shape[-1]

    with torch.no_grad():

        outputs = llm_model.generate(
            **inputs,
            max_new_tokens = 300,
            do_sample = False
        )

    generated_tokens = outputs[
        0,
        input_length:
    ]

    response = tokenizer.decode(generated_tokens, skip_special_tokens = True)

    return response.strip()

In [31]:
def mirx_v2(query, top_k = TOP_K):

    total_start = time.perf_counter()

    # -----------------------------------------
    # Query embedding + retrieval
    # -----------------------------------------

    retrieval_start = time.perf_counter()

    results = retrieve_v2(query, top_k = top_k)

    retrieval_time = (time.perf_counter() - retrieval_start)

    # -----------------------------------------
    # Context construction
    # -----------------------------------------

    context = build_context(results)

    # -----------------------------------------
    # Qwen generation
    # -----------------------------------------

    generation_start = time.perf_counter()

    answer = generate_with_qwen(query, context)

    generation_time = (time.perf_counter() - generation_start)

    total_time = (time.perf_counter()- total_start)

    return {
        "answer": answer,
        "results": results,
        "context": context,
        "retrieval_time": retrieval_time,
        "generation_time": generation_time,
        "total_time": total_time
    }

In [32]:
query = "What are the alternatives to paracetamol?"

result = mirx_v2(query, top_k = TOP_K)

print("=" * 80)
print("MIRx RESPONSE")
print("=" * 80)

print(result["answer"])

print("\n" + "=" * 80)
print("LATENCY")
print("=" * 80)

print(f"Retrieval: "f"{result['retrieval_time'] * 1000:.2f} ms")

print(f"Generation: "f"{result['generation_time']:.2f} s")

print(f"Total: "f"{result['total_time']:.2f} s")

MIRx RESPONSE
The alternatives to paracetamol (P-Aminophenol Derivative) mentioned in the retrieved information include:

- Tifmol Oral Suspension
- Moltrex Oral Suspension
- Kyomol Suspension
- Nettmol 250mg Oral Suspension
- Padicaf 250mg Oral Suspension
- Answell 400 mg/325 mg Tablet
- Bruace 400 mg/325 mg Tablet
- Rupar 400 mg/325 mg Tablet
- Brufamol Tablet
- Zupar 400mg/325mg Tablet
- Ladex-P Syrup
- Emfort P Syrup
- Mefnoc P Syrup
- Mefnix P Syrup
- Parafen Syrup
- Mefcad P Syrup
- Multigon Tablet
- Ibuwin 400 mg/500 mg Tablet
- Tolfen Tablet
- Ibuflam 400 mg/500 mg Tablet
- Arden Plus 400 mg/500 mg Tablet

LATENCY
Retrieval: 1303.48 ms
Generation: 22.56 s
Total: 23.86 s


In [33]:
print("=" * 80)
print("RETRIEVED SOURCES")
print("=" * 80)

for i, result in enumerate(result["results"]):

    print(f"\n[{i + 1}] " f"Similarity: {result.score:.4f}")

    print(result.payload["document"])

RETRIEVED SOURCES

[1] Similarity: 0.6817
name: dr best paracetamol 250 oral suspension Chemical Class: P-Aminophenol Derivative Habit Forming: No Therapeutic Class: PAIN ANALGESICS Action Class: Analgesic & Antipyretic-PCM Substitutes: Tifmol Oral Suspension, Moltrex Oral Suspension, Kyomol Suspension, Nettmol 250mg Oral Suspension, Padicaf 250mg Oral Suspension Side Effects: Indigestion, Stomach pain, Nausea, Vomiting Uses: Pain relief, Treatment of Fever

[2] Similarity: 0.6807
name: parafen forte tablet Habit Forming: No Therapeutic Class: PAIN ANALGESICS Substitutes: Answell 400 mg/325 mg Tablet, Bruace 400 mg/325 mg Tablet, Rupar 400 mg/325 mg Tablet, Brufamol Tablet, Zupar 400mg/325mg Tablet Side Effects: Heartburn, Indigestion, Nausea, Stomach pain Uses: Pain relief, Treatment of Fever

[3] Similarity: 0.6806
name: brumol m syrup Habit Forming: No Therapeutic Class: PAIN ANALGESICS Substitutes: Ladex-P Syrup, Emfort P Syrup, Mefnoc P Syrup, Mefnix P Syrup, Parafen Syrup Side Ef

In [34]:
# ============================================================
# Multiple-query evaluation
# ============================================================

test_queries = [
    "What are the alternatives to paracetamol?",
    "What are the side effects of aspirin?",
    "Which medicines are used for hypertension?",
    "What are the substitutes for ibuprofen?",
    "What drugs belong to the antibiotic class?"
]

evaluation_results = []

for query in test_queries:

    print("=" * 80)
    print("QUERY:", query)

    result = mirx_v2(
        query,
        top_k=5
    )

    print("\nANSWER:")
    print(result["answer"])

    evaluation_results.append({
        "query": query,
        "retrieval_time_ms":
            result["retrieval_time"] * 1000,
        "generation_time_s":
            result["generation_time"],
        "total_time_s":
            result["total_time"]
    })

QUERY: What are the alternatives to paracetamol?

ANSWER:
The alternatives to paracetamol (P-Aminophenol Derivative) mentioned in the retrieved information include:

- Tifmol Oral Suspension
- Moltrex Oral Suspension
- Kyomol Suspension
- Nettmol 250mg Oral Suspension
- Padicaf 250mg Oral Suspension
- Answell 400 mg/325 mg Tablet
- Bruace 400 mg/325 mg Tablet
- Rupar 400 mg/325 mg Tablet
- Brufamol Tablet
- Zupar 400mg/325mg Tablet
- Ladex-P Syrup
- Emfort P Syrup
- Mefnoc P Syrup
- Mefnix P Syrup
- Parafen Syrup
- Mefcad P Syrup
- Multigon Tablet
- Ibuwin 400 mg/500 mg Tablet
- Tolfen Tablet
- Ibuflam 400 mg/500 mg Tablet
- Arden Plus 400 mg/500 mg Tablet
QUERY: What are the side effects of aspirin?

ANSWER:
The side effects of aspirin mentioned in the retrieved information include:
- Heartburn
- Increased bleeding tendency
- Nausea
- Upset stomach
- Vomiting
QUERY: Which medicines are used for hypertension?

ANSWER:
The following medicines are used for hypertension based on the provi

In [35]:
# ============================================================
# V2 benchmark summary
# ============================================================

evaluation_df = pd.DataFrame(evaluation_results)

display(evaluation_df)

print("\nAverage retrieval latency:", evaluation_df["retrieval_time_ms"].mean())

print("Average generation latency:", evaluation_df["generation_time_s"].mean())

print("Average total latency:", evaluation_df["total_time_s"].mean())

,query,retrieval_time_ms,generation_time_s,total_time_s
0,What are the alternatives to paracetamol?,1272.6865,17.489151,18.761856
1,What are the side effects of aspirin?,554.8782,2.429459,2.984363
2,Which medicines are used for hypertension?,516.8964,3.374448,3.891366
3,What are the substitutes for ibuprofen?,499.6279,2.712465,3.212109
4,What drugs belong to the antibiotic class?,514.3086,6.480549,6.994894



Average retrieval latency: 671.6795200001798
Average generation latency: 6.497214419999727
Average total latency: 7.168917480000164
